# Custom Spam Detection Model
This notebook implements a **Custom Multinomial Naive Bayes** classifier from scratch and applies it to the Enron email dataset.

## 1. Imports

In [ ]:
import pandas as pd
import re
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

## 2. Custom Multinomial Naive Bayes Implementation
The following class implements the Naive Bayes logic, including Laplace smoothing and log-scale probability calculations.

In [ ]:
class MyMultinomialNB:
    def __init__(self, alpha=1.0):
        self.alpha = alpha
        self.classes_ = None
        self.class_log_prior_ = {}
        self.feature_log_prob_ = {}

    def fit(self, X, y):
        y = np.array(y)
        self.classes_ = np.unique(y)
        n_samples = X.shape[0]

        for c in self.classes_:
            X_c = X[y == c]
            self.class_log_prior_[c] = np.log(X_c.shape[0] / n_samples)
            feature_count = np.asarray(X_c.sum(axis=0)).ravel()
            smoothed_feature_count = feature_count + self.alpha
            smoothed_feature_total = smoothed_feature_count.sum()
            self.feature_log_prob_[c] = np.log(smoothed_feature_count / smoothed_feature_total)
        return self

    def predict(self, X):
        predictions = []
        for i in range(X.shape[0]):
            x = X[i]
            class_scores = []
            for c in self.classes_:
                log_prior = self.class_log_prior_[c]
                log_likelihood = x.dot(self.feature_log_prob_[c])[0]
                class_scores.append(log_prior + log_likelihood)
            predictions.append(self.classes_[np.argmax(class_scores)])
        return np.array(predictions)

## 3. Data Processing Functions

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r"\W+", " ", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()

def preprocess_data(df):
    df["Message"] = df["Message"].fillna("")
    df["Subject"] = df["Subject"].fillna("")
    df["text"] = df["Subject"] + " " + df["Message"]
    df["text"] = df["text"].apply(clean_text)
    df["label"] = df["Spam/Ham"].map({"ham": 0, "spam": 1})
    df = df.dropna(subset=["label"]).copy()
    return df["text"], df["label"].astype(int)

## 4. Load, Clean, and Split

In [ ]:
data_path = "data/enron_spam_data.csv"
df_raw = pd.read_csv(data_path)
X_text, y = preprocess_data(df_raw)

X_train, X_test, y_train, y_test = train_test_split(
    X_text, y, test_size=0.2, random_state=42, stratify=y
)
print(f"Training samples: {len(X_train)}, Testing samples: {len(X_test)}")

## 5. Feature Extraction and Training

In [ ]:
vectorizer = TfidfVectorizer(max_features=5000, stop_words="english")
X_train_vec = vectorizer.fit_transform(X_train)

model = MyMultinomialNB(alpha=1.0)
model.fit(X_train_vec, y_train)
print("Model trained.")

## 6. Evaluation

In [ ]:
X_test_vec = vectorizer.transform(X_test)
y_pred = model.predict(X_test_vec)

print(f"Accuracy: {accuracy_score(y_test, y_pred)}")
print("
Classification Report:
", classification_report(y_test, y_pred))

## 7. Manual Testing

In [ ]:
def test_email(subject, message):
    text = clean_text(subject + " " + message)
    vec = vectorizer.transform([text])
    res = model.predict(vec)[0]
    return "SPAM" if res == 1 else "HAM"

print("Test 1:", test_email("Free Prize", "Claim your rewards now!"))
print("Test 2:", test_email("Lunch?", "Are we still on for lunch today?"))